# Low-Cost Indoor Air Quality Monitoring and Short-Term Forecasting on ESP32 Using a Humidity-Compensated MOS Sensor and TinyML

Full training pipeline (Google Colab).

**Hardware:** ESP32 + Winsen MP-135 module (AO → GPIO 33) + DHT22 + SSD1306 OLED, sampling every 30 s.

**Pipeline**
1. Load and clean the raw logger data
2. Sensor physics: V_out → Rs → Rs/R0 ratio
3. Humidity/temperature compensation (fitted on the calibration week)
4. Relative Pollution Index (rolling 3-hour baseline) and 4 classes
5. Future targets (2 / 5 / 10 min ahead)
6. Exploratory figures
7. Features
8. Development test: day-by-day expanding window, 23–29 Oct
9. **Held-out test: 30 Oct – 1 Nov (evaluated once)**
10. Early-warning (event onset) test
11. Explainability (feature importance + SHAP)
12. TinyML export → `aq_model.h` for the ESP32
13. Save the filled dataset

**Data split**

| Part | Dates | Used for |
|---|---|---|
| Warm-up | 14–15 Oct | excluded (sensor pre-heat ≥ 48 h) |
| Calibration week | 16–22 Oct | fitting R0, compensation, class thresholds |
| Development | 23–29 Oct | choosing the method |
| Held-out test | 30 Oct – 1 Nov | final, unbiased evaluation |

## 0. Setup

In [ ]:
!pip -q install lightgbm xgboost m2cgen shap openpyxl

In [ ]:
import os, warnings, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb, xgboost as xgb
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, accuracy_score,
                             balanced_accuracy_score, f1_score, confusion_matrix, roc_auc_score,
                             average_precision_score, roc_curve, precision_recall_curve)
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 10})
SEED = 0
RUN_SLOW_MODELS = True   # set False to skip RandomForest / XGBoost / MLP classifiers (faster)
FIG_DIR = 'paper_figures'; os.makedirs(FIG_DIR, exist_ok=True)
def save_fig(name):
    # save the current figure as 300-dpi PNG + vector PDF for the paper, then show it
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/{name}.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'{FIG_DIR}/{name}.pdf', bbox_inches='tight')
    plt.show()

## 1. Upload the dataset from your PC
Choose how to load the data with `DATA_SOURCE`:

| `DATA_SOURCE` | What happens |
|---|---|
| `'upload'` (default) | a **Choose Files** button appears, pick the dataset from your computer |
| `'drive'` | reads the file from your Google Drive (`DRIVE_PATH`) |
| `'path'` | uses a file already in the Colab session (`DATA_PATH`) |

Accepted: `.xlsx`, `.xls` or `.csv`, either `Air_Quality_Dataset.xlsx` (clean) or the raw Google-Sheets export. Only the measured columns are used; everything else is recomputed.

In [ ]:
DATA_SOURCE = 'upload'                                       # 'upload' | 'drive' | 'path'
DRIVE_PATH  = '/content/drive/MyDrive/Air_Quality_Dataset.xlsx'
DATA_PATH   = 'Air_Quality_Dataset.xlsx'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if os.environ.get('AQ_DATA_PATH'):                           # used only for offline testing
    DATA_SOURCE, DATA_PATH = 'path', os.environ['AQ_DATA_PATH']

if DATA_SOURCE == 'upload':
    if not IN_COLAB:
        raise RuntimeError("Upload works in Google Colab. Outside Colab set DATA_SOURCE = 'path'.")
    from google.colab import files
    print('Click "Choose Files" and select the dataset (.xlsx / .xls / .csv) from your PC ...')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No file was uploaded - run this cell again.')
    DATA_PATH = next(iter(uploaded))
    print(f'Uploaded: {DATA_PATH} ({len(uploaded[DATA_PATH]) / 1e6:.1f} MB)')
elif DATA_SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = DRIVE_PATH

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f'{DATA_PATH} not found')
ext = os.path.splitext(DATA_PATH)[1].lower()
raw = pd.read_csv(DATA_PATH) if ext == '.csv' else pd.read_excel(DATA_PATH)
missing = [c for c in ['Time', 'Temp C', 'Humidity %', 'gas_adc'] if c not in raw.columns]
if missing:
    raise ValueError(f'Required columns missing: {missing}. Found: {list(raw.columns)}')
raw['Time'] = pd.to_datetime(raw['Time'])
raw = raw.sort_values('Time').reset_index(drop=True)
# works with the raw Google-Sheets export OR the clean dataset: keep only the measured columns, recompute everything else
RAW_COLS = ['Time', 'Temp C', 'Humidity %', 'gas_adc', 'gas_delta', 'humidity_delta', 'temp_delta', 'rolling_mean', 'rolling_std']
raw = raw[[c for c in RAW_COLS if c in raw.columns]]
print(raw.shape, raw.Time.min(), '->', raw.Time.max())
raw.head()

## 2. Configuration

In [ ]:
VC, VREF, ADC_MAX = 5.0, 3.3, 4095     # sensor circuit voltage, ESP32 ADC full scale, 12-bit ADC
RL_KOHM = 1.0                          # module load resistor (assumed; cancels in every ratio)
WARMUP_END = pd.Timestamp('2026-10-16')
CAL_END    = pd.Timestamp('2026-10-23')
TEST_START = pd.Timestamp('2026-10-30')
DEV_DAYS   = pd.date_range('2026-10-23', '2026-10-29')
SMOOTH, BASELINE = '150s', '3h'        # 2.5-min smoothing, 3-hour rolling baseline
HORIZONS = {2: 4, 5: 10, 10: 20}      # minutes -> rows (30 s per row); 10 min = main target (deployed on ESP32)
CLASS_NAMES = ['Normal', 'Slightly Elevated', 'Elevated', 'Spike']

## 3. Cleaning and quality flags

In [ ]:
d = raw.copy()
dt = d.Time.diff().dt.total_seconds()
d['flag_warmup']    = (d.Time < WARMUP_END).astype(int)
d['flag_duplicate'] = (dt < 5).fillna(False).astype(int)        # double uploads
d['flag_saturated'] = (d.gas_adc >= 4095).astype(int)           # ADC clipped
d['flag_after_gap'] = (dt > 120).fillna(False).astype(int)
ok = d.gas_adc.notna() & (d.gas_adc > 0)
print(d[['flag_warmup','flag_duplicate','flag_saturated','flag_after_gap']].sum())
print('rows per day:'); print(d.groupby(d.Time.dt.date).size().to_string())

## 4. Sensor physics: V_out → Rs → ratio
Rs = (Vc / V_RL − 1) × RL (MP-135 datasheet). R0 = 75th percentile of Rs in the calibration week (no clean-air session was recorded).

In [ ]:
d['v_out'] = np.where(ok, d.gas_adc / ADC_MAX * VREF, np.nan)
d['rs'] = (VC / d.v_out - 1) * RL_KOHM
cal = (d.Time >= WARMUP_END) & (d.Time < CAL_END) & (d.flag_duplicate == 0) & ok
R0 = float(np.nanpercentile(d.loc[cal, 'rs'], 75))
d['r0'] = R0
d['ratio'] = d.rs / R0
print(f'R0 = {R0:.4f} kOhm')

## 5. Humidity / temperature compensation
log(ratio) = a + b·Humidity + c·Temperature, fitted on the calibration week only.

In [ ]:
H = d['Humidity %'].ffill().bfill(); T = d['Temp C'].ffill().bfill()
comp_lr = LinearRegression().fit(np.c_[H[cal], T[cal]], np.log(d.loc[cal, 'ratio']))
COMP_A, (COMP_BH, COMP_BT) = comp_lr.intercept_, comp_lr.coef_
COMP_R2 = comp_lr.score(np.c_[H[cal], T[cal]], np.log(d.loc[cal, 'ratio']))
d['ratio_comp'] = d.ratio / np.exp(comp_lr.predict(np.c_[H, T]))
print(f'log(ratio) = {COMP_A:.4f} + ({COMP_BH:.5f})*H + ({COMP_BT:.5f})*T   R2 = {COMP_R2:.3f} (calibration week)')

# does the humidity effect persist? (paper table)
rows = []
for name, a, b in [('calibration 16-22 Oct', WARMUP_END, CAL_END), ('development 23-29 Oct', CAL_END, TEST_START), ('held-out 30 Oct-1 Nov', TEST_START, d.Time.max() + pd.Timedelta('1s'))]:
    m = ok & (d.flag_duplicate == 0) & (d.Time >= a) & (d.Time < b); y = np.log(d.ratio[m])
    refit = LinearRegression().fit(np.c_[H[m], T[m]], y)
    rows.append(dict(period=name, rows=int(m.sum()), corr_humidity=np.corrcoef(H[m], y)[0, 1],
                     R2_frozen_model=r2_score(y, comp_lr.predict(np.c_[H[m], T[m]])), R2_refit_same_period=refit.score(np.c_[H[m], T[m]], y),
                     humidity_coef_refit=refit.coef_[0]))
humidity_table = pd.DataFrame(rows); humidity_table.round(3)

## 6. Relative Pollution Index, classes and future targets
Index = 100 × [2.5-min mean of 1/ratio_comp] ÷ [median of 1/ratio_comp over the previous 3 h]. Class thresholds = 75th / 95th / 99th percentiles of the calibration week.

In [ ]:
v = d[(d.flag_duplicate == 0) & ok].set_index('Time')
inv = 1 / v.ratio_comp                                   # higher = more gas
rel = inv.rolling(SMOOTH).mean() / inv.rolling(BASELINE, closed='left').median()
rel[v.index < WARMUP_END] = np.nan
P75, P95, P99 = np.nanpercentile(rel[(v.index >= WARMUP_END) & (v.index < CAL_END)], [75, 95, 99])
def to_class(r): return np.select([r > P99, r > P95, r > P75], [3, 2, 1], 0)
cls = pd.Series(to_class(rel), index=v.index).where(rel.notna())
print(f'thresholds: P75={P75:.4f}  P95={P95:.4f}  P99={P99:.4f}')

out = pd.DataFrame(index=v.index)
out['Pollution Index'] = 100 * rel
out['current_class'] = cls
tser = pd.Series(v.index, index=v.index)
for mins, n in HORIZONS.items():
    col = 'target_label' if mins == 10 else f'target_{mins}m'
    out[col] = cls.shift(-n)
    out.loc[(tser.shift(-n) - tser).dt.total_seconds() > n * 30 * 1.5, col] = np.nan   # horizon crosses a gap
d = d.drop(columns=[c for c in ['Pollution Index', 'target_label'] if c in d]).merge(out.reset_index(), on='Time', how='left')
d['Air Quality'] = d.current_class.map(dict(enumerate(CLASS_NAMES)))
d['split'] = np.select([d.Time < WARMUP_END, d.Time < CAL_END, d.Time < TEST_START], ['warmup', 'calibration', 'development'], 'held-out test')
print(d['Air Quality'].value_counts(normalize=True).round(3))

## 7. Exploratory figures

In [ ]:
e = d[d.Time >= WARMUP_END].set_index('Time')
fig, ax = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
ax[0].plot(d.Time, d.gas_adc, lw=0.4); ax[0].axvspan(d.Time.min(), WARMUP_END, color='grey', alpha=0.2, label='warm-up'); ax[0].set_ylabel('gas_adc'); ax[0].legend()
ax[1].plot(d.Time, d['Humidity %'], lw=0.5, color='tab:green'); ax[1].set_ylabel('Humidity %')
ax[2].plot(d.Time, d['Pollution Index'], lw=0.4, color='tab:red'); ax[2].set_ylabel('Pollution Index')
for a in ax:
    a.axvline(CAL_END, ls='--', c='k', lw=0.8); a.axvline(TEST_START, ls='--', c='k', lw=0.8)
plt.suptitle('Raw sensor signal, humidity and relative Pollution Index'); save_fig('Fig03_timeseries_overview')

hr = e.groupby(e.index.hour)
fig, ax1 = plt.subplots(figsize=(8, 3.5)); ax2 = ax1.twinx()
ax1.plot(hr.gas_adc.median(), 'o-', label='gas_adc (median)'); ax2.plot(hr['Humidity %'].median(), 's-', c='tab:green', label='humidity (median)')
ax1.set_xlabel('hour of day'); ax1.set_ylabel('gas_adc'); ax2.set_ylabel('humidity %'); plt.title('Diurnal pattern of sensor signal and humidity'); fig.legend(loc='upper center', ncol=2); save_fig('Fig04_diurnal_pattern')

s = e[e.ratio.notna()].sample(5000, random_state=SEED)
plt.figure(figsize=(5.5, 4)); plt.scatter(s['Humidity %'], np.log(s.ratio), s=3, alpha=0.3)
plt.xlabel('Humidity %'); plt.ylabel('log(Rs/R0)'); plt.title(f'Humidity effect on Rs/R0 (calibration-week R2 = {COMP_R2:.2f})'); save_fig('Fig05_humidity_vs_ratio')

## 8. Features (all use past/current data only)

In [ ]:
D = d[(d.flag_duplicate == 0) & d.current_class.notna()].set_index('Time')
r = D['Pollution Index'] / 100
X = pd.DataFrame(index=D.index)
X['idx'] = r
for k in [1, 2, 5, 10, 20]: X[f'idx_lag{k}'] = r.shift(k)
X['idx_d5'] = r - r.shift(5); X['idx_d10'] = r - r.shift(10)
X['ratio_comp'] = D.ratio_comp; X['ratio_comp_d10'] = D.ratio_comp - D.ratio_comp.shift(10)
X['gas_std10'] = D.gas_adc.rolling(10).std() / D.gas_adc.rolling(10).mean()
X['temp'] = D['Temp C'].ffill(); X['hum'] = D['Humidity %'].ffill(); X['hum_d10'] = X.hum - X.hum.shift(10)
h = D.index.hour + D.index.minute / 60
X['hour_sin'] = np.sin(2 * np.pi * h / 24); X['hour_cos'] = np.cos(2 * np.pi * h / 24)
X['cur'] = D.current_class
for w in [20, 60, 120]: X[f'idx_max{w}'] = r.rolling(w).max(); X[f'idx_mean{w}'] = r.rolling(w).mean()
FEATURES = list(X.columns); print(len(FEATURES), 'features:', FEATURES)

### Models
Key idea: the regression models predict the **change** in the index (future − now). Persistence ("future = now") is the baseline every model must beat.

In [ ]:
def lgb_change(small=False):
    p = dict(n_estimators=60, num_leaves=8, learning_rate=0.1) if small else dict(n_estimators=300, num_leaves=15, learning_rate=0.03)
    return lgb.LGBMRegressor(min_child_samples=100, objective='l1', verbose=-1, random_state=SEED, **p)
REG_MODELS = {'Ridge (change)': lambda: make_pipeline(StandardScaler(), Ridge(1.0)),
              'LightGBM-300 (change)': lambda: lgb_change(False),
              'TinyML LightGBM-60 (change)': lambda: lgb_change(True)}
def CLS_MODELS():
    m = {'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
         'LightGBM': lgb.LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=50, verbose=-1, random_state=SEED)}
    if RUN_SLOW_MODELS:
        m['Random Forest'] = RandomForestClassifier(300, min_samples_leaf=10, n_jobs=-1, random_state=SEED)
        m['XGBoost'] = xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=SEED, verbosity=0)
        m['MLP'] = make_pipeline(StandardScaler(), MLPClassifier((64, 32), max_iter=60, early_stopping=True, random_state=SEED))
    return m
L = [0, 1, 2, 3]
def cls_metrics(y, p):
    pc = f1_score(y, p, average=None, labels=L, zero_division=0)
    return dict(accuracy=accuracy_score(y, p), balanced_acc=balanced_accuracy_score(y, p), macro_f1=f1_score(y, p, average='macro', labels=L, zero_division=0),
                f1_normal=pc[0], f1_slight=pc[1], f1_elevated=pc[2], f1_spike=pc[3], n=len(y))
def dataset(mins):
    n = HORIZONS[mins]; col = 'target_label' if mins == 10 else f'target_{mins}m'
    yr = r.shift(-n); yc = D[col]
    m = X.notna().all(1) & yr.notna() & yc.notna() & (X.index >= WARMUP_END)
    return X[m], yr[m], yc[m].astype(int)

## 9. Development test (23–29 Oct): each day is predicted by models trained only on earlier days

In [ ]:
dev_reg, dev_cls = [], []
for mins in HORIZONS:
    Xm, yr, yc = dataset(mins)
    Xm, yr, yc = Xm[Xm.index < TEST_START], yr[yr.index < TEST_START], yc[yc.index < TEST_START]
    R = {k: [] for k in ['Persistence', *REG_MODELS]}; Cp = {k: [] for k in ['Persistence', 'TinyML -> class', *CLS_MODELS()]}; Yr, Yc = [], []
    for day in DEV_DAYS:
        tr = Xm.index < day; te = (Xm.index >= day) & (Xm.index < day + pd.Timedelta('1D'))
        base = Xm.idx[te].values; Yr.append(yr[te].values); Yc.append(yc[te].values)
        R['Persistence'].append(base); Cp['Persistence'].append(Xm.cur[te].astype(int).values)
        for k, f in REG_MODELS.items():
            p = base + f().fit(Xm[tr], (yr - Xm.idx)[tr]).predict(Xm[te]); R[k].append(p)
            if k.startswith('TinyML'): Cp['TinyML -> class'].append(to_class(p))
        for k, mdl in CLS_MODELS().items(): Cp[k].append(mdl.fit(Xm[tr], yc[tr]).predict(Xm[te]))
    yrt, yct = np.concatenate(Yr), np.concatenate(Yc); pm = mean_absolute_error(yrt, np.concatenate(R['Persistence']))
    for k, v in R.items():
        v = np.concatenate(v); dev_reg.append(dict(horizon_min=mins, model=k, MAE_points=100 * mean_absolute_error(yrt, v), improvement_pct=100 * (1 - mean_absolute_error(yrt, v) / pm)))
    for k, v in Cp.items(): dev_cls.append(dict(horizon_min=mins, model=k, **cls_metrics(yct, np.concatenate(v))))
    print('done', mins, 'min')
dev_reg = pd.DataFrame(dev_reg); dev_cls = pd.DataFrame(dev_cls)
display(dev_reg.round(3)); display(dev_cls.round(3))

## 10. Held-out test (30 Oct – 1 Nov) — trained on 16–29 Oct, evaluated once

In [ ]:
test_reg, test_cls, store = [], [], {}
for mins in HORIZONS:
    Xm, yr, yc = dataset(mins)
    tr, te = Xm.index < TEST_START, Xm.index >= TEST_START
    base = Xm.idx[te].values; yrt = yr[te].values; yct = yc[te].values
    preds = {'Persistence': base}
    for k, f in REG_MODELS.items(): preds[k] = base + f().fit(Xm[tr], (yr - Xm.idx)[tr]).predict(Xm[te])
    pm = mean_absolute_error(yrt, base)
    for k, p in preds.items():
        test_reg.append(dict(horizon_min=mins, model=k, MAE_points=100 * mean_absolute_error(yrt, p), RMSE_points=100 * np.sqrt(mean_squared_error(yrt, p)),
                             improvement_pct=100 * (1 - mean_absolute_error(yrt, p) / pm), R2=r2_score(yrt, p), n=len(yrt)))
    cp = {'Always Normal': np.zeros(len(yct), int), 'Persistence': Xm.cur[te].astype(int).values,
          'TinyML -> class': to_class(preds['TinyML LightGBM-60 (change)'])}
    for k, mdl in CLS_MODELS().items(): cp[k + ' (direct)'] = mdl.fit(Xm[tr], yc[tr]).predict(Xm[te])
    for k, p in cp.items(): test_cls.append(dict(horizon_min=mins, model=k, **cls_metrics(yct, p)))
    store[mins] = dict(time=Xm.index[te], y=yrt, base=base, tiny=preds['TinyML LightGBM-60 (change)'], yc=yct, pc=cp['TinyML -> class'], pers=cp['Persistence'])
test_reg = pd.DataFrame(test_reg); test_cls = pd.DataFrame(test_cls)
print('Held-out test: Pollution Index forecast'); display(test_reg.round(3))
print('Held-out test: class forecast'); display(test_cls.round(3))

In [ ]:
s10 = store[10]
fig, ax = plt.subplots(1, 3, figsize=(15, 4), gridspec_kw={'width_ratios': [2, 1, 1]})
k = slice(0, 2880)   # first day of the test
ax[0].plot(s10['time'][k], 100 * s10['y'][k], lw=0.8, label='actual (t+10 min)')
ax[0].plot(s10['time'][k], 100 * s10['base'][k], lw=0.8, alpha=0.7, label='persistence')
ax[0].plot(s10['time'][k], 100 * s10['tiny'][k], lw=0.8, label='TinyML forecast')
ax[0].set_ylabel('Pollution Index'); ax[0].legend(); ax[0].set_title('10-min forecast on a held-out day')
for a, key, title in [(ax[1], 'pers', 'Persistence'), (ax[2], 'pc', 'TinyML')]:
    cm = confusion_matrix(s10['yc'], s10[key], labels=L)
    a.imshow(cm, cmap='Blues'); a.set_xticks(L); a.set_yticks(L); a.set_xticklabels(['N', 'S', 'E', 'Sp']); a.set_yticklabels(['N', 'S', 'E', 'Sp'])
    for i in L:
        for j in L: a.text(j, i, cm[i, j], ha='center', va='center', fontsize=8)
    a.set_xlabel('predicted'); a.set_ylabel('true'); a.set_title(f'{title} confusion matrix (10 min)'); a.grid(False)
save_fig('Fig10_forecast_example_and_confusion')

t = test_reg[test_reg.model != 'Persistence']
plt.figure(figsize=(7, 3.5))
for i, m in enumerate(t.model.unique()):
    q = t[t.model == m]; plt.bar(np.arange(3) + i * 0.27, q.improvement_pct, width=0.27, label=m)
plt.xticks(np.arange(3) + 0.27, [f'{h} min' for h in HORIZONS]); plt.ylabel('MAE improvement vs persistence (%)'); plt.axhline(0, c='k', lw=0.8)
plt.title('Held-out forecast improvement over persistence'); plt.legend(fontsize=8); save_fig('Fig11_improvement_vs_persistence')

## 11. Early warning: from a Normal/Slightly Elevated state, will an Elevated/Spike state occur within N minutes?

In [ ]:
onset = []; cur = D.current_class; pr_store = {}
plt.figure(figsize=(5, 4))
for mins in [5, 10]:
    n = HORIZONS[mins]
    fut = (cur >= 2).astype(float)[::-1].rolling(n, min_periods=n).max()[::-1].shift(-1)
    m = X.notna().all(1) & fut.notna() & (X.cur < 2) & (X.index >= WARMUP_END)
    Xb, yb = X[m], fut[m].astype(int); tr, te = Xb.index < TEST_START, Xb.index >= TEST_START
    scores = {'Baseline (current index)': Xb.idx[te].values,
              'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xb[tr], yb[tr]).predict_proba(Xb[te])[:, 1],
              'LightGBM': lgb.LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=50, verbose=-1, random_state=SEED).fit(Xb[tr], yb[tr]).predict_proba(Xb[te])[:, 1]}
    for k, sc in scores.items():
        onset.append(dict(horizon_min=mins, model=k, roc_auc=roc_auc_score(yb[te], sc), pr_auc=average_precision_score(yb[te], sc), positives=int(yb[te].sum()), n=int(te.sum())))
        if mins == 5:
            fpr, tpr, _ = roc_curve(yb[te], sc); plt.plot(fpr, tpr, label=f'{k} (AUC {roc_auc_score(yb[te], sc):.2f})')
            pr_store[k] = (yb[te].values, sc)
plt.plot([0, 1], [0, 1], 'k--', lw=0.7); plt.xlabel('false positive rate'); plt.ylabel('true positive rate'); plt.title('5-min early warning ROC (held-out)'); plt.legend(fontsize=8); save_fig('Fig14_early_warning_roc')
onset = pd.DataFrame(onset); onset.round(3)

## 11b. Spray-event analysis (ground truth)
All events classified as **Spike** were confirmed by the occupant to be room air-freshener sprays. This section lists every event, shows their typical shape, and measures forecast quality during spray onset, during the decay back to normal, and during normal periods (held-out test).

In [ ]:
os.makedirs('paper_tables', exist_ok=True)
sp_idx = D.index[D.current_class == 3]; events = []; s0 = pv = None
for t in sp_idx:                                   # group Spike rows into events (gaps < 10 min merged)
    if s0 is None: s0 = pv = t; continue
    if t - pv > pd.Timedelta('10min'): events.append((s0, pv)); s0 = t
    pv = t
if s0 is not None: events.append((s0, pv))
rows = []
for s_, e_ in events:
    seg = D.loc[s_:e_]; pk = seg['Pollution Index'].idxmax()
    before = D.loc[s_ - pd.Timedelta('15min'): s_ - pd.Timedelta('30s'), 'gas_adc'].median()
    back = D.loc[e_:].index[D.loc[e_:].current_class == 0]
    rows.append(dict(event=len(rows) + 1, start=s_, peak_time=pk, peak_index=seg['Pollution Index'].max(), peak_gas_adc=int(seg.gas_adc.max()),
                     usual_gas_adc=before, rise_factor=seg.gas_adc.max() / before, spike_minutes=(e_ - s_).total_seconds() / 60 + 0.5,
                     minutes_to_normal=(back[0] - pk).total_seconds() / 60 if len(back) else np.nan,
                     humidity=D.loc[s_, 'Humidity %'], split=d.set_index('Time').split.reindex([s_]).iloc[0]))
spray = pd.DataFrame(rows)
spray.round(2).to_csv('paper_tables/Table7_spray_events.csv', index=False)
print(f'{len(spray)} spray events | median peak index {spray.peak_index.median():.0f} | median rise x{spray.rise_factor.median():.2f} | '
      f'median Spike duration {spray.spike_minutes.median():.1f} min | median time back to Normal {spray.minutes_to_normal.median():.0f} min')
print(spray.split.value_counts().to_string())
spray.round(2)

In [ ]:
# --- Fig 18: all spray events aligned to their start
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
grid = np.arange(-10, 91, 0.5); curves = []
for _, ev_ in spray.iterrows():
    w = D.loc[ev_.start - pd.Timedelta('10min'): ev_.start + pd.Timedelta('90min'), 'Pollution Index']
    if len(w) < 20: continue
    tmin = (w.index - ev_.start).total_seconds() / 60
    yv = np.interp(grid, tmin, w.values, left=np.nan, right=np.nan); curves.append(yv)
    ax[0].plot(grid, yv, c='grey', lw=0.6, alpha=0.5)
med = np.nanmedian(np.vstack(curves), axis=0)
ax[0].plot(grid, med, c='tab:red', lw=2.2, label='median of all events')
for th, lab in [(P75, 'Slightly Elevated'), (P95, 'Elevated'), (P99, 'Spike')]: ax[0].axhline(100 * th, ls=':', c='k', lw=0.7)
ax[0].axvline(0, c='k', lw=0.8); ax[0].set_ylim(80, 260); ax[0].set_xlabel('minutes from spray'); ax[0].set_ylabel('Pollution Index')
ax[0].set_title(f'{len(curves)} room-spray events aligned to start'); ax[0].legend()
ax[1].hist(spray.minutes_to_normal.dropna(), bins=15, color='tab:blue', alpha=0.8)
ax[1].axvline(spray.minutes_to_normal.median(), c='tab:red', lw=2, label=f'median {spray.minutes_to_normal.median():.0f} min')
ax[1].set_xlabel('minutes from peak back to Normal'); ax[1].set_ylabel('events'); ax[1].set_title('Recovery time after spraying'); ax[1].legend()
save_fig('Fig18_spray_events_aligned')

In [ ]:
# --- Fig 19 + Table 8: forecast error by phase (held-out test, 10-min horizon)
s10 = store[10]; tgt = pd.DatetimeIndex(s10['time']) + pd.Timedelta('10min')     # time each forecast refers to
phase = np.full(len(tgt), 'Normal periods', dtype=object)
for _, ev_ in spray[spray.split == 'held-out test'].iterrows():
    phase[(tgt >= ev_.start - pd.Timedelta('2min')) & (tgt <= ev_.peak_time)] = 'Spray onset (rise to peak)'
    rec = max(ev_.minutes_to_normal if pd.notna(ev_.minutes_to_normal) else 10, 10)
    phase[(tgt > ev_.peak_time) & (tgt <= ev_.peak_time + pd.Timedelta(minutes=rec))] = 'After spray (decay to normal)'
ph_rows = []
for ph in ['Spray onset (rise to peak)', 'After spray (decay to normal)', 'Normal periods']:
    k = phase == ph
    mp, mt = 100 * mean_absolute_error(s10['y'][k], s10['base'][k]), 100 * mean_absolute_error(s10['y'][k], s10['tiny'][k])
    ph_rows.append(dict(phase=ph, rows=int(k.sum()), MAE_persistence=mp, MAE_TinyML=mt, improvement_pct=100 * (1 - mt / mp)))
phase_table = pd.DataFrame(ph_rows); phase_table.round(2).to_csv('paper_tables/Table8_error_by_phase.csv', index=False)
x_ = np.arange(len(phase_table))
plt.figure(figsize=(7.5, 3.8))
plt.bar(x_ - 0.2, phase_table.MAE_persistence, 0.4, label='Persistence'); plt.bar(x_ + 0.2, phase_table.MAE_TinyML, 0.4, label='TinyML')
for i, v in enumerate(phase_table.improvement_pct): plt.text(i, max(phase_table.MAE_persistence[i], phase_table.MAE_TinyML[i]) + 1, f'{v:+.1f}%', ha='center', fontsize=9)
plt.xticks(x_, ['Spray onset', 'After spray\n(decay)', 'Normal\nperiods']); plt.ylabel('MAE (index points)')
plt.title('10-min forecast error by phase (held-out test)'); plt.legend(); save_fig('Fig19_error_by_phase')
phase_table.round(2)

## 12. Explainability

In [ ]:
Xm, yr, yc = dataset(10)
tiny_all_dev = lgb_change(True).fit(Xm[Xm.index < TEST_START], (yr - Xm.idx)[Xm.index < TEST_START])
imp = pd.Series(tiny_all_dev.booster_.feature_importance('gain'), index=FEATURES); imp = (imp / imp.sum()).sort_values()
imp.plot.barh(figsize=(6, 6), title='TinyML feature importance (gain share)'); save_fig('Fig16_feature_importance')
try:
    import shap
    sample = Xm[Xm.index >= TEST_START].sample(min(2000, int((Xm.index >= TEST_START).sum())), random_state=SEED)
    sv = shap.TreeExplainer(tiny_all_dev).shap_values(sample)
    shap.summary_plot(sv, sample, show=False, max_display=12); plt.title('SHAP values, held-out sample'); save_fig('Fig17_shap_summary')
except Exception as ex:
    print('SHAP skipped:', ex)

## 12b. Additional figures for the paper
System architecture, methodology flow, data coverage, class distribution, correlations, sensor drift, compensation effect, an example pollution event, model comparison, error vs horizon, predicted vs actual, residuals and the precision-recall curve. All figures are saved to `paper_figures/` as 300-dpi PNG and PDF.

In [ ]:
from matplotlib.patches import FancyBboxPatch
def box(ax, x, y, w, h, text, color):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.02', fc=color, ec='k', lw=1))
    ax.text(x + w / 2, y + h / 2, text, ha='center', va='center', fontsize=8.5)
def arrow(ax, x1, y1, x2, y2): ax.annotate('', (x2, y2), (x1, y1), arrowprops=dict(arrowstyle='->', lw=1.2))
# --- Fig 1: system architecture
fig, ax = plt.subplots(figsize=(11, 4.2)); ax.set_xlim(0, 11); ax.set_ylim(0, 4.2); ax.axis('off')
box(ax, 0.2, 2.6, 2.0, 1.0, 'MP-135 MOS gas sensor\n(AO, 0.1-4.0 V)', '#fde2c8')
box(ax, 0.2, 1.1, 2.0, 1.0, 'DHT22\ntemperature & humidity', '#fde2c8')
box(ax, 3.0, 0.5, 4.4, 3.4, '', '#dbe9f6')
ax.text(5.2, 3.65, 'ESP32 (every 30 s)', ha='center', fontsize=10, weight='bold')
steps = ['12-bit ADC -> V_out -> Rs -> Rs/R0', 'Humidity/temperature compensation', 'Relative Pollution Index (3-h baseline)', 'Current class (4 levels)', 'TinyML LightGBM -> +10 min forecast']
for i, t in enumerate(steps): box(ax, 3.25, 3.0 - i * 0.55, 3.9, 0.42, t, 'white')
box(ax, 8.2, 2.6, 2.5, 1.0, 'SSD1306 OLED\nlive class + forecast', '#d9f2d9')
box(ax, 8.2, 1.1, 2.5, 1.0, 'Wi-Fi -> Google Apps Script\n-> Google Sheets', '#d9f2d9')
arrow(ax, 2.2, 3.1, 3.0, 3.0); arrow(ax, 2.2, 1.6, 3.0, 1.9); arrow(ax, 7.4, 2.6, 8.2, 3.1); arrow(ax, 7.4, 1.9, 8.2, 1.6)
plt.title('System architecture of the low-cost ESP32 air quality monitor'); save_fig('Fig01_system_architecture')

# --- Fig 2: methodology flow
fig, ax = plt.subplots(figsize=(12, 2.6)); ax.set_xlim(0, 12); ax.set_ylim(0, 2.6); ax.axis('off')
flow = ['Data logging\n30 s, 19 days', 'Cleaning\nwarm-up, duplicates,\nsaturation, gaps', 'Sensor physics\nRs, R0, ratio', 'Humidity/temp\ncompensation', 'Relative index\n+ 4 classes', 'Future targets\n2/5/10 min', 'Models vs\npersistence', 'Held-out test\n+ TinyML export']
for i, t in enumerate(flow):
    box(ax, 0.1 + i * 1.48, 0.7, 1.3, 1.2, t, '#eef3fb' if i % 2 else '#fff4e5')
    if i: arrow(ax, 0.1 + i * 1.48 - 0.18, 1.3, 0.1 + i * 1.48, 1.3)
plt.title('Methodology'); save_fig('Fig02_methodology_flow')

# --- Fig 6: data coverage per day
cnt = d.groupby(d.Time.dt.date).size(); colors = d.groupby(d.Time.dt.date).split.first().map({'warmup': 'grey', 'calibration': 'tab:blue', 'development': 'tab:orange', 'held-out test': 'tab:red'})
plt.figure(figsize=(10, 3.2)); plt.bar(range(len(cnt)), cnt.values, color=colors.values)
plt.axhline(2880, ls='--', c='k', lw=0.8, label='expected (2880/day)')
plt.xticks(range(len(cnt)), [str(x)[5:] for x in cnt.index], rotation=60); plt.ylabel('readings per day')
for lab, c in [('warm-up', 'grey'), ('calibration', 'tab:blue'), ('development', 'tab:orange'), ('held-out test', 'tab:red')]: plt.bar([], [], color=c, label=lab)
plt.legend(fontsize=8, ncol=5); plt.title('Data coverage per day'); save_fig('Fig06_data_coverage')

# --- Fig 7: class distribution per split
cd = pd.crosstab(d.split, d['Air Quality'], normalize='index').reindex(index=['calibration', 'development', 'held-out test'], columns=CLASS_NAMES) * 100
cd.plot.bar(stacked=True, figsize=(7, 3.8), color=['#4caf50', '#ffc107', '#ff7043', '#b71c1c'])
plt.ylabel('% of readings'); plt.xticks(rotation=0); plt.legend(fontsize=8, bbox_to_anchor=(1, 1)); plt.title('Air quality class distribution by data split'); save_fig('Fig07_class_distribution')

# --- Fig 8: correlation heatmap
cols = ['gas_adc', 'ratio', 'ratio_comp', 'Pollution Index', 'Temp C', 'Humidity %']
cm_ = e[cols].corr()
fig, ax = plt.subplots(figsize=(6, 5)); im = ax.imshow(cm_, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(cols))); ax.set_yticks(range(len(cols))); ax.set_xticklabels(cols, rotation=45, ha='right'); ax.set_yticklabels(cols)
for i in range(len(cols)):
    for j in range(len(cols)): ax.text(j, i, f'{cm_.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)
ax.grid(False); plt.colorbar(im, fraction=0.046); plt.title('Correlation matrix (16 Oct - 1 Nov)'); save_fig('Fig08_correlation_heatmap')

# --- Fig 9: drift + compensation effect
day = e.resample('D').agg({'ratio': 'median', 'Humidity %': 'median'})
hr_ = e.groupby(e.index.hour).agg({'ratio': 'median', 'ratio_comp': 'median', 'Pollution Index': 'median'})
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(day.index, day.ratio, 'o-', label='daily median Rs/R0'); a2 = ax[0].twinx(); a2.plot(day.index, day['Humidity %'], 's--', c='tab:green', label='daily median humidity')
ax[0].set_ylabel('Rs/R0'); a2.set_ylabel('humidity %'); ax[0].set_title('Day-to-day baseline drift'); ax[0].tick_params(axis='x', rotation=45)
ax[1].plot(hr_.index, hr_.ratio / hr_.ratio.mean(), 'o-', label='raw Rs/R0')
ax[1].plot(hr_.index, hr_.ratio_comp / hr_.ratio_comp.mean(), 's-', label='compensated Rs/R0')
ax[1].plot(hr_.index, hr_['Pollution Index'] / hr_['Pollution Index'].mean(), '^-', label='Pollution Index')
ax[1].set_xlabel('hour of day'); ax[1].set_ylabel('normalised (mean = 1)'); ax[1].legend(fontsize=8); ax[1].set_title('Diurnal cycle removed by compensation + baseline')
save_fig('Fig09_drift_and_compensation')

In [ ]:
# --- Fig 12: example pollution event with classes and forecast (largest held-out event)
s10 = store[10]; ts = pd.Series(100 * s10['y'], index=s10['time'] + pd.Timedelta('10min'))   # actual index at its own time
peak = ts.idxmax(); w0, w1 = peak - pd.Timedelta('50min'), peak + pd.Timedelta('70min')
seg = e.loc[w0:w1]
fc = pd.Series(100 * s10['tiny'], index=s10['time'] + pd.Timedelta('10min')).loc[w0:w1]   # forecast plotted at the time it refers to
fig, ax = plt.subplots(2, 1, figsize=(10, 5), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
ax[0].plot(seg.index, seg['Pollution Index'], lw=1.5, label='Pollution Index (actual)')
ax[0].plot(fc.index, fc.values, lw=1.2, ls='--', label='TinyML forecast issued 10 min earlier')
for th, lab in [(P75, 'Slightly Elevated'), (P95, 'Elevated'), (P99, 'Spike')]: ax[0].axhline(100 * th, ls=':', lw=0.8, c='grey'); ax[0].text(seg.index[0], 100 * th + 1, lab, fontsize=7, color='grey')
ax[0].set_ylabel('index'); ax[0].legend(fontsize=8); ax[0].set_title(f'Example pollution event, {peak:%d %b %H:%M}')
ax[1].plot(seg.index, seg.gas_adc, c='tab:purple', lw=1); ax[1].set_ylabel('gas_adc')
save_fig('Fig12_example_event')

# --- Fig 13: model comparison (macro F1) and error vs horizon
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
pv = test_cls.pivot(index='model', columns='horizon_min', values='macro_f1').sort_values(10)
pv.plot.barh(ax=ax[0]); ax[0].set_xlabel('macro F1 (held-out)'); ax[0].set_title('Class forecast: model comparison'); ax[0].legend(title='horizon (min)', fontsize=8)
for m in test_reg.model.unique():
    q = test_reg[test_reg.model == m]; ax[1].plot(q.horizon_min, q.MAE_points, 'o-', label=m)
ax[1].set_xlabel('forecast horizon (min)'); ax[1].set_ylabel('MAE (index points)'); ax[1].set_title('Forecast error vs horizon (held-out)'); ax[1].legend(fontsize=8)
save_fig('Fig13_model_comparison_and_horizon')

# --- Fig 15: predicted vs actual + residuals (10 min)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
lim = [np.percentile(100 * s10['y'], 0.5), np.percentile(100 * s10['y'], 99.8)]
ax[0].scatter(100 * s10['y'], 100 * s10['tiny'], s=3, alpha=0.3); ax[0].plot(lim, lim, 'k--', lw=0.8)
ax[0].set_xlim(lim); ax[0].set_ylim(lim); ax[0].set_xlabel('actual index (t+10 min)'); ax[0].set_ylabel('TinyML forecast'); ax[0].set_title('Predicted vs actual (held-out)')
ax[1].hist(100 * (s10['y'] - s10['base']), bins=120, range=(-40, 40), alpha=0.6, label='persistence')
ax[1].hist(100 * (s10['y'] - s10['tiny']), bins=120, range=(-40, 40), alpha=0.6, label='TinyML')
ax[1].set_xlabel('error (index points)'); ax[1].set_ylabel('count'); ax[1].set_yscale('log'); ax[1].legend(); ax[1].set_title('Error distribution (10 min)')
save_fig('Fig15_pred_vs_actual_and_residuals')

# --- Fig 14b: precision-recall for early warning
plt.figure(figsize=(5, 4))
for k, (y_, sc) in pr_store.items():
    pr, rc, _ = precision_recall_curve(y_, sc); plt.plot(rc, pr, label=f'{k} (AP {average_precision_score(y_, sc):.2f})')
plt.axhline(np.mean(list(pr_store.values())[0][0]), ls='--', c='k', lw=0.7, label='random')
plt.xlabel('recall'); plt.ylabel('precision'); plt.legend(fontsize=8); plt.title('5-min early warning precision-recall (held-out)'); save_fig('Fig14b_early_warning_pr')

### Paper tables (CSV)
Table 1 hardware cost is filled in by hand; the other tables are generated from the results.

In [ ]:
TAB_DIR = 'paper_tables'; os.makedirs(TAB_DIR, exist_ok=True)
pd.DataFrame([['ESP32 DevKit', 'microcontroller, Wi-Fi', ''], ['Winsen MP-135 module', 'MOS gas sensor', 590], ['DHT22', 'temperature & humidity', ''], ['SSD1306 0.96" OLED', 'display', ''], ['Wires, breadboard, USB adapter', '', '']],
             columns=['Component', 'Role', 'Price (BDT)']).to_csv(f'{TAB_DIR}/Table1_hardware_cost_FILL_IN.csv', index=False)
stats = d[d.split != 'warmup'].groupby('split').agg(rows=('Time', 'size'), start=('Time', 'min'), end=('Time', 'max'),
        gas_adc_median=('gas_adc', 'median'), temp_mean=('Temp C', 'mean'), humidity_mean=('Humidity %', 'mean'))
stats.to_csv(f'{TAB_DIR}/Table2_dataset_summary.csv')
humidity_table.to_csv(f'{TAB_DIR}/Table3_humidity_effect.csv', index=False)
test_reg.to_csv(f'{TAB_DIR}/Table4_heldout_index_forecast.csv', index=False)
test_cls.to_csv(f'{TAB_DIR}/Table5_heldout_class_forecast.csv', index=False)
onset.to_csv(f'{TAB_DIR}/Table6_heldout_early_warning.csv', index=False)
dev_reg.to_csv(f'{TAB_DIR}/TableS1_development_index_forecast.csv', index=False)
dev_cls.to_csv(f'{TAB_DIR}/TableS2_development_class_forecast.csv', index=False)
pd.DataFrame({'feature': FEATURES}).to_csv(f'{TAB_DIR}/TableS3_feature_list.csv', index=False)
display(stats)

## 13. TinyML export for the ESP32
The deployed model is re-trained on **all** labelled rows (16 Oct – 1 Nov) with the same settings, then converted to plain C with `m2cgen`. Copy `aq_model.h` into the Arduino sketch folder and the constants below into the `.ino` file.

In [ ]:
import m2cgen as m2c
Xa, yra, _ = dataset(10)
final_model = lgb_change(True).fit(Xa.values, (yra - Xa.idx).values)
c_code = m2c.export_to_c(final_model, function_name='aq_model_delta')
header = ('// Auto-generated LightGBM model (60 trees, 8 leaves), trained on %d rows.\n' % len(Xa) +
          '// Predicts the CHANGE in Pollution Index/100 ten minutes ahead.\n// Input order:\n' +
          ''.join(f'//   [{i:2d}] {f}\n' for i, f in enumerate(FEATURES)) + '#pragma once\n#include <math.h>\n')
open('aq_model.h', 'w').write(header + c_code)
print(f'aq_model.h written: {len(c_code)/1024:.1f} KB of C code')
consts = dict(R0_KOHM=R0, COMP_A=COMP_A, COMP_BH=COMP_BH, COMP_BT=COMP_BT, TH_P75=P75, TH_P95=P95, TH_P99=P99)
print('\n// paste into AirQualityLive.ino')
for k, val in consts.items(): print(f'const double {k:8s} = {float(val)!r};')

## 14. Save results and the filled dataset

In [ ]:
keep = ['Time', 'Temp C', 'Humidity %', 'Pollution Index', 'Air Quality', 'gas_adc', 'rs', 'r0', 'ratio', 'gas_delta', 'humidity_delta', 'temp_delta',
        'rolling_mean', 'rolling_std', 'target_2m', 'target_5m', 'target_label', 'current_class', 'v_out', 'ratio_comp', 'split',
        'flag_warmup', 'flag_duplicate', 'flag_saturated', 'flag_after_gap']
with pd.ExcelWriter('Air_Quality_Filled_and_Results.xlsx') as w:
    d[keep].to_excel(w, sheet_name='Data', index=False)
    test_reg.to_excel(w, sheet_name='Test Forecast (Index)', index=False)
    test_cls.to_excel(w, sheet_name='Test Classes', index=False)
    onset.to_excel(w, sheet_name='Test Early Warning', index=False)
    humidity_table.to_excel(w, sheet_name='Humidity Effect', index=False)
    dev_reg.to_excel(w, sheet_name='Dev Forecast', index=False)
    dev_cls.to_excel(w, sheet_name='Dev Classes', index=False)
print('saved Air_Quality_Filled_and_Results.xlsx and aq_model.h')
import shutil
os.system('zip -qr paper_figures_and_tables.zip paper_figures paper_tables')
print(sorted(os.listdir(FIG_DIR))); print(sorted(os.listdir('paper_tables')))
try:
    from google.colab import files
    for f in ['Air_Quality_Filled_and_Results.xlsx', 'aq_model.h', 'paper_figures_and_tables.zip']: files.download(f)
except ImportError:
    pass